In [4]:
!pip install pandas scikit-learn numpy matplotlib seaborn
import pandas as pd
import numpy as np

# 1. 讀取週五下午的 PortScan 資料集
# 因為檔案很大，我們先用 nrows=200000 讀取前 20 萬筆，讓跑的速度變快
file_path = 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv'
df = pd.read_csv(file_path, nrows=200000)

# 2. 清除欄位名稱前後包含的空白字元（這是這個資料集經典的坑）
df.columns = df.columns.str.strip()

# 3. 將資料集中的無限大值 (inf) 替換為 NaN，然後直接刪除這些少數異常列
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# 4. 檢查並印出目前的資料標籤分佈情況
print("【資料標籤分佈】")
print(df['Label'].value_counts())


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


【資料標籤分佈】
Label
PortScan    109268
BENIGN       90478
Name: count, dtype: int64


In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import pandas as pd
import numpy as np

print("【第一步：開始挑選網路行為特徵...】")
feature_cols = [
    'Flow Duration',          
    'Total Fwd Packets',      
    'Total Backward Packets', 
    'Flow Bytes/s',           
    'Flow Packets/s'          
]

# 1. 強制轉數值並補值
X = df[feature_cols].apply(pd.to_numeric, errors='coerce')
X = X.fillna(X.mean())

# 2. 特徵標準化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. 確保資料型態為標準的 float32，避免新版 sklearn 報錯
X_scaled = np.asarray(X_scaled, dtype=np.float32)
print("特徵矩陣處理完成，形狀為：", X_scaled.shape)


print("\n【第二步：開始訓練 Isolation Forest 模型...】")
# 4. 部署孤立森林模型（拿掉 n_jobs 參數，避開新版本的執行緒相容性檢查）
clf = IsolationForest(contamination=0.5, random_state=42)

# 5. 【修正核心】將 fit 與 predict 拆開執行，這是最穩定的寫法
clf.fit(X_scaled)                  # 先訓練模型
predicted_codes = clf.predict(X_scaled) # 再進行預測

# 6. 將結果寫回 DataFrame 中
df['Predicted_Code'] = predicted_codes
df['Anomaly_Predicted'] = df['Predicted_Code'].map({1: 'BENIGN', -1: 'Anomaly'})

print("AI 模型預測完成！")

【第一步：開始挑選網路行為特徵...】
特徵矩陣處理完成，形狀為： (199746, 5)

【第二步：開始訓練 Isolation Forest 模型...】
AI 模型預測完成！


In [10]:
import pandas as pd

# 使用 pandas 的 crosstab 功能，把「原始正確答案(Label)」跟「AI預測結果(Anomaly_Predicted)」做交叉對比
evaluation = pd.crosstab(df['Label'], df['Anomaly_Predicted'])

print("【AI 惡意流量偵測 - 成果評估表】")
print("---------------------------------------")
print(evaluation)
print("---------------------------------------")

# 額外算一下簡單的準確率
total_samples = len(df)
# 看看有沒有成功對齊
print(f"總共分析了 {total_samples} 筆網路流量。")

【AI 惡意流量偵測 - 成果評估表】
---------------------------------------
Anomaly_Predicted  Anomaly  BENIGN
Label                             
BENIGN               82299    8179
PortScan             17230   92038
---------------------------------------
總共分析了 199746 筆網路流量。


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

print("【第一步：準備監督式學習的標籤...】")
# 1. 隨機森林需要正確答案。我們將 Label 轉為二進位：BENIGN 轉為 0，PortScan 轉為 1
y = df['Label'].map({'BENIGN': 0, 'PortScan': 1})

# 2. 切分訓練集與測試集 (80% 訓練模型，20% 留著驗證 AI 聰不聰明)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f"訓練集大小: {X_train.shape}, 測試集大小: {X_test.shape}")


print("\n【第二步：開始訓練隨機森林分類器 (Random Forest)...】")
# 3. 初始化隨機森林模型
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# 4. 丟入有標準答案的資料進行強監督訓練
rf_clf.fit(X_train, y_train)
print("模型訓練完成！")


print("\n【第三步：驗證預測成效...】")
# 5. 讓訓練好的 AI 預測那 20% 的未知測試集
y_pred = rf_clf.predict(X_test)

# 6. 印出資安最核心的評估指標與混淆矩陣
print("--- 混淆矩陣 (Confusion Matrix) ---")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=['真實 BENIGN', '真實 PortScan'], columns=['AI預測正常', 'AI預測異常'])
print(cm_df)

print("\n--- 詳細分類報告 (Classification Report) ---")
print(classification_report(y_test, y_pred, target_names=['BENIGN', 'PortScan']))

【第一步：準備監督式學習的標籤...】
訓練集大小: (159796, 5), 測試集大小: (39950, 5)

【第二步：開始訓練隨機森林分類器 (Random Forest)...】
模型訓練完成！

【第三步：驗證預測成效...】
--- 混淆矩陣 (Confusion Matrix) ---
             AI預測正常  AI預測異常
真實 BENIGN     17525     571
真實 PortScan      40   21814

--- 詳細分類報告 (Classification Report) ---
              precision    recall  f1-score   support

      BENIGN       1.00      0.97      0.98     18096
    PortScan       0.97      1.00      0.99     21854

    accuracy                           0.98     39950
   macro avg       0.99      0.98      0.98     39950
weighted avg       0.99      0.98      0.98     39950

